In [0]:
%run ./00_common

In [0]:
%run ./02_delta_writer

In [0]:
%run ./03_logger_utils

In [0]:
%run ./05_notifier

In [0]:
import pandas as pd

def create_alert_record(spark, run_id, alert_type, severity, message):
    alert_pdf = pd.DataFrame([{
        "alert_ts": utc_now_naive(),
        "run_id": run_id,
        "alert_type": alert_type,
        "severity": severity,
        "message": message
    }])
    append_alerts_delta(alert_pdf)

def notify_failure(
    spark,
    run_id,
    stage,
    dataset,
    error_message,
    error_code,
    smtp_user="",
    smtp_app_password="",
    to_email=""
):
    create_alert_record(
        spark=spark,
        run_id=run_id,
        alert_type="PIPELINE_FAILURE",
        severity="HIGH",
        message=f"[{stage}/{dataset}] {error_message}"
    )

    log_event(
        spark=spark,
        level="ERROR",
        run_id=run_id,
        stage=stage,
        dataset=dataset,
        status="FAILED",
        message=error_message,
        error_code=error_code
    )

    if smtp_user and smtp_app_password and to_email:
        subject = f"[ELT LAB V2 PF] Falla en pipeline | stage={stage} | dataset={dataset} | run_id={run_id}"
        body = f"""
Se detectó una falla en el pipeline ELT V2 pandas-first.

Run ID: {run_id}
Stage: {stage}
Dataset: {dataset}
Error code: {error_code}
Mensaje: {error_message}

Revisa en Databricks:
- {OPS_PIPELINE_EVENTS}
- {OPS_PIPELINE_ALERTS}
- logs del notebook
""".strip()

        sent, email_error = try_send_failure_email(
            smtp_user=smtp_user,
            smtp_app_password=smtp_app_password,
            to_email=to_email,
            subject=subject,
            body=body
        )

        if sent:
            log_event(
                spark=spark,
                level="INFO",
                run_id=run_id,
                stage=stage,
                dataset=dataset,
                status="ALERT_SENT",
                message=f"Correo enviado a {to_email}"
            )
        else:
            create_alert_record(
                spark=spark,
                run_id=run_id,
                alert_type="EMAIL_FAILURE",
                severity="MEDIUM",
                message=f"No se pudo enviar correo: {email_error}"
            )
            log_event(
                spark=spark,
                level="WARN",
                run_id=run_id,
                stage=stage,
                dataset=dataset,
                status="EMAIL_FAILED",
                message=f"No se pudo enviar correo: {email_error}",
                error_code="EMAIL_001"
            )